In [1]:
import json, os, h5py, numpy as np
from collections import Counter

DATA = "data"

with open(f"{DATA}/vg150/VG-SGG-dicts.json") as f:
    d = json.load(f)
idx_label = {int(k): v for k, v in d['idx_to_label'].items()}
idx_pred = {int(k): v for k, v in d['idx_to_predicate'].items()}

with h5py.File(f"{DATA}/vg150/VG-SGG.h5", "r") as f:
    splits = f['split'][:]
    boxes_1024 = f['boxes_1024'][:]
    boxes_512 = f['boxes_512'][:]
    labels = f['labels'][:].flatten()
    rels = f['relationships'][:]
    preds = f['predicates'][:].flatten()
    first_b = f['img_to_first_box'][:]
    last_b = f['img_to_last_box'][:]
    first_r = f['img_to_first_rel'][:]
    last_r = f['img_to_last_rel'][:]
    img_to_first = f['img_to_first_box'][:]
    img_to_last = f['img_to_last_box'][:]
    active_mask = f.get('active_object_mask', None)
    if active_mask: active_mask = active_mask[:].flatten()

n_imgs = len(img_to_first)
print(f"{len(boxes_1024)} boxes, {len(rels)} rels, {n_imgs} images")
print(f"splits: train={(splits==0).sum()} val={(splits==1).sum()} test={(splits==2).sum()}")


1145398 boxes, 622705 rels, 108073 images
splits: train=75651 val=0 test=32422


In [2]:
# box size stats
w = boxes_1024[:, 2]
h = boxes_1024[:, 3]
areas = w * h
print(f"width:  min={w.min():.0f} max={w.max():.0f} mean={w.mean():.0f}")
print(f"height: min={h.min():.0f} max={h.max():.0f} mean={h.mean():.0f}")
print(f"area:   min={areas.min():.0f} max={areas.max():.0f} mean={areas.mean():.0f}")
print(f"boxes < 100px: {(areas < 100).sum()} ({(areas < 100).sum()/len(boxes_1024)*100:.2f}%)")
print(f"zero-area boxes: {(areas == 0).sum()}")


width:  min=2 max=1024 mean=244
height: min=2 max=1024 mean=221
area:   min=58 max=1048576 mean=80581
boxes < 100px: 11 (0.00%)
zero-area boxes: 0


In [3]:
# relationship stats
n_imgs = len(first_r)
img_rels = last_r - first_r + 1
print(f"total relationships: {len(rels)}")
print(f"per image: {len(rels)/n_imgs:.1f} avg")
print(f"images with 0 rels: {(img_rels == 0).sum()}")
print(f"max rels in one image: {img_rels.max()}")


total relationships: 622705
per image: 5.8 avg
images with 0 rels: 0
max rels in one image: 272


In [4]:
# memory footprint
total = 0
for key in ['boxes_1024','boxes_512','labels','relationships','predicates','split','img_to_first_box','img_to_last_box','img_to_first_rel','img_to_last_rel','active_object_mask']:
    ds_data = None
    if key == 'boxes_1024': ds_data = boxes_1024
    elif key == 'boxes_512': ds_data = boxes_512
    elif key == 'labels': ds_data = labels.reshape(-1, 1) if labels.ndim == 1 else labels
    elif key == 'relationships': ds_data = rels
    elif key == 'predicates': ds_data = preds.reshape(-1, 1) if preds.ndim == 1 else preds
    elif key == 'split': ds_data = splits.reshape(-1, 1) if splits.ndim == 1 else splits
    elif key == 'img_to_first_box': ds_data = first_b.reshape(-1, 1) if first_b.ndim == 1 else first_b
    elif key == 'img_to_last_box': ds_data = last_b.reshape(-1, 1) if last_b.ndim == 1 else last_b
    elif key == 'img_to_first_rel': ds_data = first_r.reshape(-1, 1) if first_r.ndim == 1 else first_r
    elif key == 'img_to_last_rel': ds_data = last_r.reshape(-1, 1) if last_r.ndim == 1 else last_r
    elif key == 'active_object_mask' and active_mask is not None:
        ds_data = active_mask.reshape(-1, 1) if active_mask.ndim == 1 else active_mask
    if ds_data is not None:
        size = np.prod(ds_data.shape) * ds_data.dtype.itemsize
        if isinstance(ds_data, np.ndarray):
            size = ds_data.nbytes
        total += size
        print(f"{key:25s} {str(ds_data.shape):15s} {size/1024**2:.1f} MB")
print(f"\ntotal: {total/1024**2:.1f} MB")


boxes_1024                (1145398, 4)    17.5 MB
boxes_512                 (1145398, 4)    17.5 MB
labels                    (1145398, 1)    8.7 MB
relationships             (622705, 2)     4.8 MB
predicates                (622705, 1)     4.8 MB
split                     (108073, 1)     0.4 MB
img_to_first_box          (108073, 1)     0.4 MB
img_to_last_box           (108073, 1)     0.4 MB
img_to_first_rel          (108073, 1)     0.4 MB
img_to_last_rel           (108073, 1)     0.4 MB
active_object_mask        (1145398, 1)    1.1 MB

total: 56.3 MB


In [5]:
# coordinate range check
xc, yc, w, h = boxes_1024[:,0], boxes_1024[:,1], boxes_1024[:,2], boxes_1024[:,3]
x1 = xc - w/2
y1 = yc - h/2
x2 = xc + w/2
y2 = yc + h/2
print(f"x1 range: {x1.min():.0f} to {x1.max():.0f}")
print(f"y1 range: {y1.min():.0f} to {y1.max():.0f}")
print(f"negative x1: {(x1 < 0).sum()}, negative y1: {(y1 < 0).sum()}")
print(f"x2 > 1024: {(x2 > 1024).sum()}, y2 > 1024: {(y2 > 1024).sum()}")


x1 range: -0 to 1022
y1 range: -0 to 1022
negative x1: 36728, negative y1: 33786
x2 > 1024: 0, y2 > 1024: 0


In [6]:
# class distribution
cnt = Counter(labels)
print("top 20 classes:")
for cid, count in cnt.most_common(20):
    name = idx_label.get(cid, f"?{cid}")
    print(f"  {name:20s} {count:6d}")
print(f"\ntotal unique: {len(cnt)}")


top 20 classes:
  man                   54659
  tree                  49901
  window                42444
  person                37924
  shirt                 33794
  building              31740
  woman                 25053
  sign                  23488
  leg                   22326
  head                  21313
  pole                  21181
  table                 19053
  hand                  17362
  car                   17333
  hair                  17173
  light                 14112
  leaf                  13745
  people                13475
  door                  13304
  pant                  13091

total unique: 150


In [7]:
# objects per image
counts = last_b - first_b + 1
print(f"objects per image: min={counts.min()}, max={counts.max()}, mean={counts.mean():.1f}")
print(f"images with 0 objects: {(counts <= 0).sum()}")
print(f"images with 1 object: {(counts == 1).sum()}")
print(f"images with 50+ objects: {(counts >= 50).sum()}")


objects per image: min=1, max=62, mean=10.6
images with 0 objects: 0
images with 1 object: 4569
images with 50+ objects: 14


In [8]:
# predicate distribution
cnt = Counter(preds)
max_count = cnt.most_common(1)[0][1]
print("top 25 predicates:")
for pid, count in cnt.most_common(25):
    name = idx_pred.get(pid, f"?{pid}")
    bar = "#" * int(count / max_count * 40)
    print(f"  {name:20s} {bar} {count}")


top 25 predicates:
  on                   ######################################## 196495
  has                  ################### 97473
  wearing              ############## 71121
  of                   ########## 53761
  in                   ####### 37914
  near                 ###### 30331
  with                 ### 18428
  behind               ### 18159
  holding              ### 16514
  above                ## 11602
  sitting on           # 8003
  wears                # 7471
  under                # 6687
  riding               # 6335
  in front of          # 5609
  standing on           4068
  at                    3030
  attached to           2689
  carrying              2504
  walking on            2343
  over                  1871
  belonging to          1518
  for                   1507
  looking at            1410
  watching              1327


In [9]:
# random checks
# images with no boxes
empty = (last_b == first_b - 1).sum()
print(f"images with no boxes: {empty}")

# label range
print(f"label range: {labels.min()} to {labels.max()} (0-based)")
out_of_range = (labels >= 150).sum()
print(f"labels >= 150: {out_of_range} / {len(labels)}")

# file sizes
h5_size = os.path.getsize(f"{DATA}/vg150/VG-SGG.h5")
import glob
preproc_fs = glob.glob(f"{DATA}/preprocessed/*.npy")
preproc_size = sum(os.path.getsize(p) for p in preproc_fs) + sum(os.path.getsize(p) for p in glob.glob(f"{DATA}/preprocessed/*.pkl"))
print(f"h5: {h5_size/1024**2:.0f} MB")
print(f"preprocessed: {preproc_size/1024**2:.0f} MB")


images with no boxes: 0
label range: 1 to 150 (0-based)
labels >= 150: 4406 / 1145398
h5: 56 MB
preprocessed: 482 MB
